# Recirculation box - expert mode : Pb_Hydraulique

In [ ]:
from trustutils import run
import matplotlib.pyplot as plt
run.introduction("Edouard Butaye & Alan Burlot")
run.TRUST_parameters()

## Problem description

This form allows to compare the version 1994 and 2003 of $k-\omega$ SST model in a simple channel flow. The model is tested implicit and explicit time schemes in VDF and VEF discretisation. The computation are performed with and without production limiters.

We simulate a single phase turbulent flow in RANS in a periodic channel flow. The channel is $L=0.012\,m$ long and $H=0.66667\,m$ height. The Reynolds number based on the bulk velocity is $7500$.


## Definition of bloc dictionnaries
### Discretisations
The domain uses one block. Both VEF and VDF are tested. For VEF, we use :
- Nx = 7, Ny = 101

For VDF, we use
- Nx = 13, Ny = 201

The grid is refined at the wall. The first mesh is 2mm height, ensuring $y^+<1$ at the wall.

In [ ]:
dic_VEF = {"discretisation": "VEF",
           "triangulate": "trianguler_H dom",
           "nx": 7,
           "ny": 101}
dic_VDF = {"discretisation": "VDF",
           "triangulate": "",
           "nx": 13,
           "ny": 201}

### Time schemes

Both explicit and implicit Euler schemes are tested. For the implicit scheme, a `facsec = 20` is used.

In [ ]:
dic_expl = {
    "scheme": "schema_euler_explicite",
    "scheme_options":
    """
        tinit 0.
        tmax 2
        dt_min 1e-7
        facsec 1
        dt_impr 1
        seuil_statio 1.e-6
    """
}
dic_impl = {
    "scheme": "schema_euler_implicite",
    "scheme_options":
    """
        tinit 0.
        tmax 2
        dt_min 1e-7
        facsec 1
        facsec_max 20
        dt_impr 1
        seuil_statio 1.e-6
        solveur implicite { solveur gmres { diag nb_it_max 3 seuil 1e-12 impr } }
    """
}

WARNING: 10s of computation is not enough for the velocity profil to converge.

### Computation setup

For the $k-\omega$ model, we perform a test for both SST and STD variants.

In [ ]:
ddis = {
    "VEF": dic_VEF,
    "VDF": dic_VDF
}
dscheme = {
    "EXPL": dic_expl,
    "IMPL": dic_impl
}

dic_activate = {"production_limiters": ""}
dic_deactivate = {"production_limiters": "deactivate_production_limiter"}
dproduction_limiters = {
    "with_prod_limit": dic_activate,
    "without_prod_limit": dic_deactivate
}
dic_menter_1994 = {"menter_version": "ORIGINAL_1994" }
dic_menter_2003 = {"menter_version": "MODIFIED_2003" }
dexpert_mode = {
"Menter_1994": dic_menter_1994,
"Menter_2003": dic_menter_2003,
}

### Post-processing

In [ ]:
run.reset()
run.initBuildDirectory()
# K-omega cases
for kscheme, vscheme in dscheme.items():
    for kdis, vdis in ddis.items():
        for kexpert_mode,vexpert_mode in dexpert_mode.items():
            for kprod,vprod in dproduction_limiters.items():
                target_repo = f"{kdis}/{kscheme}/{kexpert_mode}/{kprod}"
                run.addCaseFromTemplate("recirculation_box_expert_mode.data", target_repo, {**vdis, **vscheme, **vexpert_mode,**vprod})

run.printCases()


In [ ]:
run.runCases()

### Performances

In [ ]:
table = run.tablePerf()
table = table.drop(columns=["host", "system"]).drop("Total")
table

## Results
### Residuals

In [ ]:
from trustutils import plot
import itertools

marker = itertools.cycle(('^', '+', 'd', 'o', '*', '<', '>'))

dcolors_k_eps = {"VEF": "blue", "VDF": "red"}
dcolors_k_omega = {"VEF": "cyan", "VDF": "orange"}

dstyle = {"IMPL": "solid", "EXPL": "dashed"}

In [ ]:
nb_cases=len(ddis.keys())*len(dscheme.keys())*len(dexpert_mode.keys())*len(dproduction_limiters.keys())
cmap = plt.cm.get_cmap('Paired', nb_cases)
colors = cmap(range(nb_cases))

a = plot.Graph("Residuals with wall law")
ind_color=0
for kscheme, vscheme in dscheme.items():
    for kdis, vdis in ddis.items():
        for kexpert_mode,vexpert_mode in dexpert_mode.items():
            for kprod,vprod in dproduction_limiters.items():
                a.addResidu(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/{kexpert_mode}/{kprod}/recirculation_box_expert_mode.dt_ev",
                            label=f"{kdis} {kscheme} {kexpert_mode} {kprod}",
                            color=colors[ind_color])
                ind_color+=1


a.scale(yscale='log')

In [ ]:
import numpy as np
y_dns_lamballais,vx_dns_lamballais,vy_dns_lamballais=np.loadtxt("src/velocity_profile_dns_Lamballais_2014.dat",delimiter=" ",unpack=True)

In [ ]:
a = plot.Graph("Vitesse X - Euler explicite")
for kdis, vdis in ddis.items():
    for kexpert_mode,vexpert_mode in dexpert_mode.items():
        for kprod,vprod in dproduction_limiters.items():
            a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/EXPL/{kexpert_mode}/{kprod}/recirculation_box_expert_mode_VELOCITY_PROBE.son",
             compo=0,
             label=f"{kdis} {kexpert_mode} {kprod}",
             marker = next(marker),
             ls="--",
             lw=1)
u_out=0.125
plt.plot(y_dns_lamballais,vx_dns_lamballais*u_out, label="DNS Lamballais 2014") #scale to compare (extract from the backward_facing_step) : U_out lamballais = 1m/s. U_out_present_work=0.125


In [ ]:
a = plot.Graph("Velocity X - Euler implicite")
for kdis, vdis in ddis.items():
    for kexpert_mode,vexpert_mode in dexpert_mode.items():
        for kprod,vprod in dproduction_limiters.items():
            a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/IMPL/{kexpert_mode}/{kprod}/recirculation_box_expert_mode_VELOCITY_PROBE.son",
             compo=0,
             label=f"{kdis} {kexpert_mode} {kprod}",
             marker = next(marker),
             ls="--",
             lw=1)
u_out=0.125
plt.plot(y_dns_lamballais,vx_dns_lamballais*u_out, label="DNS Lamballais 2014") #scale to compare (extract from the backward_facing_step) : U_out lamballais = 1m/s. U_out_present_work=0.125


In [ ]:
a = plot.Graph("Velocity Y")

for kscheme, vscheme in dscheme.items():
    for kdis, vdis in ddis.items():
        for kexpert_mode,vexpert_mode in dexpert_mode.items():
            for kprod,vprod in dproduction_limiters.items():
                a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/IMPL/{kexpert_mode}/{kprod}/recirculation_box_expert_mode_VELOCITY_PROBE.son",
             compo=1,
             label=f"{kdis} {kscheme}{kexpert_mode} {kprod}",
             marker = next(marker),
             ls="--",
             lw=1)
                

In [ ]:
a = plot.Graph("Viscosity")

for kscheme, vscheme in dscheme.items():
    for kdis, vdis in ddis.items():
        for kexpert_mode,vexpert_mode in dexpert_mode.items():
            for kprod,vprod in dproduction_limiters.items():
                a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/{kexpert_mode}/{kprod}/recirculation_box_expert_mode_VISCOSITY_PROBE.son",
                             label=f"{kscheme} {kdis} {kexpert_mode} {kprod}", linestyle=dstyle[kscheme],
             marker = next(marker),
             lw=1)



In [ ]:
a = plot.Graph("Viscosity VDF")
kdis="VDF"
for kscheme, vscheme in dscheme.items():
    for kexpert_mode,vexpert_mode in dexpert_mode.items():
        for kprod,vprod in dproduction_limiters.items():
            a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/{kexpert_mode}/{kprod}/recirculation_box_expert_mode_VISCOSITY_PROBE.son",
                         label=f"{kscheme} {kdis} {kexpert_mode} {kprod}", linestyle=dstyle[kscheme],
         marker = next(marker),
         lw=1)


In [ ]:
a = plot.Graph("Viscosity VEF")
kdis="VEF"
for kscheme, vscheme in dscheme.items():
    for kexpert_mode,vexpert_mode in dexpert_mode.items():
        for kprod,vprod in dproduction_limiters.items():
            a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/{kexpert_mode}/{kprod}/recirculation_box_expert_mode_VISCOSITY_PROBE.son",
                         label=f"{kscheme} {kdis} {kexpert_mode} {kprod}", linestyle=dstyle[kscheme],
         marker = next(marker),
         lw=1)


In [ ]:
a = plot.Graph("K")
for kscheme, vscheme in dscheme.items():
    for kdis, vdis in ddis.items():
        for kexpert_mode,vexpert_mode in dexpert_mode.items():
            for kprod,vprod in dproduction_limiters.items():
                a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/{kexpert_mode}/{kprod}/recirculation_box_expert_mode_K_PROBE.son",
                             label=f"{kdis} {kscheme}{kexpert_mode} {kprod}", linestyle=dstyle[kscheme],marker = next(marker), lw=1)

In [ ]:
a = plot.Graph("K VDF")
kdis="VDF"
for kscheme, vscheme in dscheme.items():
        for kexpert_mode,vexpert_mode in dexpert_mode.items():
            for kprod,vprod in dproduction_limiters.items():
                a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/{kexpert_mode}/{kprod}/recirculation_box_expert_mode_K_PROBE.son",
                             label=f"{kdis} {kscheme}{kexpert_mode} {kprod}", linestyle=dstyle[kscheme],marker = next(marker), lw=1)


In [ ]:
a = plot.Graph("K VEF")
kdis="VEF"
for kscheme, vscheme in dscheme.items():
        for kexpert_mode,vexpert_mode in dexpert_mode.items():
            for kprod,vprod in dproduction_limiters.items():
                a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/{kexpert_mode}/{kprod}/recirculation_box_expert_mode_K_PROBE.son",
                             label=f"{kdis} {kscheme}{kexpert_mode} {kprod}", linestyle=dstyle[kscheme],marker = next(marker), lw=1)

In [ ]:
a = plot.Graph("Omega")
for kscheme, vscheme in dscheme.items():
    for kdis, vdis in ddis.items():
        for kexpert_mode,vexpert_mode in dexpert_mode.items():
            for kprod,vprod in dproduction_limiters.items():
                a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/{kexpert_mode}/{kprod}/recirculation_box_expert_mode_OMEGA_PROBE.son",
                             label=f"{kdis} {kscheme}{kexpert_mode} {kprod}", linestyle=dstyle[kscheme], marker = next(marker), lw=1)


In [ ]:
a = plot.Graph("Cross Diffusion")
for kscheme, vscheme in dscheme.items():
    for kdis, vdis in ddis.items():
        for kexpert_mode,vexpert_mode in dexpert_mode.items():
            for kprod,vprod in dproduction_limiters.items():
                a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/{kexpert_mode}/{kprod}/recirculation_box_expert_mode_CKOMEGA_PROBE.son",
                             label=f"{kdis} {kscheme}{kexpert_mode} {kprod}", linestyle=dstyle[kscheme], marker = next(marker), lw=1)


In [ ]:
a = plot.Graph("Production K")
for kscheme, vscheme in dscheme.items():
    for kdis, vdis in ddis.items():
        for kexpert_mode,vexpert_mode in dexpert_mode.items():
            for kprod,vprod in dproduction_limiters.items():
                a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/{kexpert_mode}/{kprod}/recirculation_box_expert_mode_PK_PROBE.son",
                             label=f"{kdis} {kscheme}{kexpert_mode} {kprod}", linestyle=dstyle[kscheme], marker = next(marker), lw=1)

In [ ]:
a = plot.Graph("Dissipation K")
for kscheme, vscheme in dscheme.items():
    for kdis, vdis in ddis.items():
        for kexpert_mode,vexpert_mode in dexpert_mode.items():
            for kprod,vprod in dproduction_limiters.items():
                a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/{kexpert_mode}/{kprod}/recirculation_box_expert_mode_DK_PROBE.son",
                             label=f"{kdis} {kscheme}{kexpert_mode} {kprod}", linestyle=dstyle[kscheme], marker = next(marker), lw=1)

In [ ]:
a = plot.Graph("Dissipation Omega")
for kscheme, vscheme in dscheme.items():
    for kdis, vdis in ddis.items():
        for kexpert_mode,vexpert_mode in dexpert_mode.items():
            for kprod,vprod in dproduction_limiters.items():
                a.addSegment(f"{run.BUILD_DIRECTORY}/{kdis}/{kscheme}/{kexpert_mode}/{kprod}/recirculation_box_expert_mode_DOMEGA_PROBE.son",
                             label=f"{kdis} {kscheme}{kexpert_mode} {kprod}", linestyle=dstyle[kscheme], marker = next(marker), lw=1)